# ID3 Algorithm - Starter Notebook
ID3 builds a decision tree top-down using Information Gain (entropy reduction) to select split attributes.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from math import log2
from collections import Counter

In [ ]:
# Classic EnjoySport dataset
columns = ['Outlook', 'Temperature', 'Humidity', 'Wind', 'PlayTennis']
data = [
    ['Sunny',    'Hot',  'High',   'Weak',   'No'],
    ['Sunny',    'Hot',  'High',   'Strong', 'No'],
    ['Overcast', 'Hot',  'High',   'Weak',   'Yes'],
    ['Rain',     'Mild', 'High',   'Weak',   'Yes'],
    ['Rain',     'Cool', 'Normal', 'Weak',   'Yes'],
    ['Rain',     'Cool', 'Normal', 'Strong', 'No'],
    ['Overcast', 'Cool', 'Normal', 'Strong', 'Yes'],
    ['Sunny',    'Mild', 'High',   'Weak',   'No'],
    ['Sunny',    'Cool', 'Normal', 'Weak',   'Yes'],
    ['Rain',     'Mild', 'Normal', 'Weak',   'Yes'],
    ['Sunny',    'Mild', 'Normal', 'Strong', 'Yes'],
    ['Overcast', 'Mild', 'High',   'Strong', 'Yes'],
    ['Overcast', 'Hot',  'Normal', 'Weak',   'Yes'],
    ['Rain',     'Mild', 'High',   'Strong', 'No'],
]
df = pd.DataFrame(data, columns=columns)
print(df.to_string(index=False))

In [ ]:
def entropy(labels):
    n = len(labels)
    if n == 0:
        return 0
    counts = Counter(labels)
    return -sum((c/n) * log2(c/n) for c in counts.values() if c > 0)

def information_gain(df, attribute, target='PlayTennis'):
    total_entropy = entropy(df[target])
    values = df[attribute].unique()
    weighted_entropy = sum(
        (len(df[df[attribute] == v]) / len(df)) * entropy(df[df[attribute] == v][target])
        for v in values
    )
    return total_entropy - weighted_entropy

features = columns[:-1]
print(f'Parent entropy: {entropy(df["PlayTennis"]):.4f}\n')
print('Information Gains:')
gains = {}
for f in features:
    ig = information_gain(df, f)
    gains[f] = ig
    print(f'  IG({f:<12}) = {ig:.4f}')
best = max(gains, key=gains.get)
print(f'\nBest split attribute: {best}')

In [ ]:
# Recursive ID3 implementation
def id3(df, features, target='PlayTennis', depth=0):
    labels = df[target].tolist()
    indent = '  ' * depth

    if len(set(labels)) == 1:
        print(f'{indent}LEAF -> {labels[0]}')
        return labels[0]
    if not features:
        majority = Counter(labels).most_common(1)[0][0]
        print(f'{indent}LEAF (no features left) -> {majority}')
        return majority

    best_attr = max(features, key=lambda f: information_gain(df, f, target))
    tree = {best_attr: {}}
    print(f'{indent}Split on: {best_attr} (IG={information_gain(df, best_attr, target):.4f})')

    for val in df[best_attr].unique():
        subset = df[df[best_attr] == val].drop(columns=[best_attr])
        remaining = [f for f in features if f != best_attr]
        print(f'{indent}  [{best_attr}={val}]')
        tree[best_attr][val] = id3(subset, remaining, target, depth + 2)

    return tree

print('=== ID3 Tree Construction ===')
tree = id3(df, features)

In [ ]:
# Bar chart of Information Gains
plt.figure(figsize=(6, 4))
plt.bar(gains.keys(), gains.values(), color='steelblue', edgecolor='k')
plt.ylabel('Information Gain')
plt.title('ID3: Information Gain per Attribute')
plt.tight_layout()
plt.show()